In [1]:
import numpy as np
import pandas as pd
import joblib

# Load encoded training/testing data
X_train_encoded = pd.read_csv("../data/processed/X_train_encoded.csv")
X_test_encoded = pd.read_csv("../data/processed/X_test_encoded.csv")

y_train_encoded = np.load("../data/processed/y_train_encoded.npy")
y_test_encoded = np.load("../data/processed/y_test_encoded.npy")

# Load scaler and target encoder
scaler = joblib.load("../data/processed/scaler.pkl")
target_encoder = joblib.load("../data/processed/target_encoder.pkl")

# Scale features
X_train_scaled = scaler.transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

print("Phase 3 setup complete.")
print("Training shape:", X_train_scaled.shape)
print("Testing shape:", X_test_scaled.shape)


Phase 3 setup complete.
Training shape: (118783, 41)
Testing shape: (29696, 41)


## Step 3.1 — Calculate Class Weights


In [8]:
from sklearn.utils.class_weight import compute_class_weight

# Compute balanced class weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_encoded),
    y=y_train_encoded
)

# Create dictionary
class_weight_dict = dict(
    enumerate(class_weights)
)

print("Class weights:")

for cls_idx, weight in class_weight_dict.items():
    print(
        f"  {target_encoder.classes_[cls_idx]}: "
        f"{weight:.2f}"
    )

Class weights:
  apache2: 23.31
  back: 4.48
  buffer_overflow: 111.64
  ftp_write: 530.28
  guess_passwd: 11.43
  httptunnel: 128.55
  imap: 385.66
  ipsweep: 1.30
  land: 265.14
  loadmodule: 385.66
  mailbomb: 56.56
  mscan: 17.11
  multihop: 326.33
  neptune: 0.11
  nmap: 3.12
  normal: 0.07
  pod: 22.45
  portsweep: 1.61
  processtable: 25.10
  rootkit: 385.66
  saint: 57.33
  satan: 1.22
  smurf: 1.66
  snmpgetattack: 124.77
  snmpguess: 53.70
  teardrop: 5.32
  warezclient: 5.32
  warezmaster: 17.75


## Step 3.2 — Controlled Oversampling

In [14]:
from imblearn.over_sampling import RandomOverSampler
from collections import Counter

print("Before oversampling:")
print(Counter(y_train_encoded))

# Target number for minority classes
max_samples = 1000

# Create oversampling strategy
oversample_strategy = {}

class_counts = Counter(y_train_encoded)

for cls, count in class_counts.items():
    if count < max_samples:
        oversample_strategy[cls] = max_samples
    else:
        oversample_strategy[cls] = count

print("\nOversampling strategy:")
print(oversample_strategy)

# Apply controlled random oversampling
ros = RandomOverSampler(
    sampling_strategy=oversample_strategy,
    random_state=42
)

X_train_resampled, y_train_resampled = ros.fit_resample(
    X_train_scaled,
    y_train_encoded
)

print("\nAfter controlled oversampling:")
print(Counter(y_train_resampled))

print("\nOriginal training shape:", X_train_scaled.shape)
print("Resampled training shape:", X_train_resampled.shape)

Before oversampling:
Counter({np.int64(15): 62870, np.int64(13): 38294, np.int64(21): 3465, np.int64(7): 3262, np.int64(17): 2642, np.int64(22): 2549, np.int64(14): 1359, np.int64(1): 946, np.int64(26): 798, np.int64(25): 797, np.int64(4): 371, np.int64(11): 248, np.int64(27): 239, np.int64(16): 189, np.int64(0): 182, np.int64(18): 169, np.int64(24): 79, np.int64(10): 75, np.int64(20): 74, np.int64(2): 38, np.int64(23): 34, np.int64(5): 33, np.int64(8): 16, np.int64(12): 13, np.int64(6): 11, np.int64(19): 11, np.int64(9): 11, np.int64(3): 8})

Oversampling strategy:
{np.int64(15): 62870, np.int64(13): 38294, np.int64(14): 1359, np.int64(5): 1000, np.int64(7): 3262, np.int64(11): 1000, np.int64(4): 1000, np.int64(18): 1000, np.int64(17): 2642, np.int64(1): 1000, np.int64(25): 1000, np.int64(2): 1000, np.int64(26): 1000, np.int64(22): 2549, np.int64(21): 3465, np.int64(27): 1000, np.int64(16): 1000, np.int64(12): 1000, np.int64(20): 1000, np.int64(0): 1000, np.int64(24): 1000, np.int64(6

## Step 3.3 — SMOTE (Only If Necessary)


In [19]:
from collections import Counter

min_class_count = min(
    Counter(y_train_resampled).values()
)

print(
    f"Minimum class count after controlled oversampling: "
    f"{min_class_count}"
)

if min_class_count < 100:
    print(
        "Consider using SMOTE for these very small classes..."
    )
else:
    print(
        "SMOTE is not necessary. "
        "Controlled oversampling has already balanced the minority classes."
    )

Minimum class count after controlled oversampling: 1000
SMOTE is not necessary. Controlled oversampling has already balanced the minority classes.


## Step 3.4 — Save Resampled Data

In [23]:
import os
import numpy as np

# Get project root based on this notebook location
project_root = os.path.abspath("..")
processed_dir = os.path.join(project_root, "data", "processed")

# Create processed folder if it does not exist
os.makedirs(processed_dir, exist_ok=True)

# Save resampled training data
X_path = os.path.join(
    processed_dir,
    "X_train_resampled.npy"
)

y_path = os.path.join(
    processed_dir,
    "y_train_resampled.npy"
)

np.save(X_path, X_train_resampled)
np.save(y_path, y_train_resampled)

print("Phase 3 complete.")
print()
print("X_train_resampled shape:", X_train_resampled.shape)
print("y_train_resampled shape:", y_train_resampled.shape)
print()
print("Saved files:")
print(X_path)
print(y_path)
print()
print("X file exists:", os.path.exists(X_path))
print("Y file exists:", os.path.exists(y_path))

Phase 3 complete.

X_train_resampled shape: (135441, 41)
y_train_resampled shape: (135441,)

Saved files:
d:\smart-cybersecurity-ml\data\processed\X_train_resampled.npy
d:\smart-cybersecurity-ml\data\processed\y_train_resampled.npy

X file exists: True
Y file exists: True


Final Processed Dataset Save

In [26]:
import os
import numpy as np
import joblib
import zipfile

# ============================================
# FINAL PROCESSED DATASET SAVE
# ============================================

processed_dir = "../data/processed"
os.makedirs(processed_dir, exist_ok=True)

# --------------------------------------------
# 1. Save resampled training data
# --------------------------------------------

np.save(
    os.path.join(
        processed_dir,
        "X_train_resampled.npy"
    ),
    X_train_resampled
)

np.save(
    os.path.join(
        processed_dir,
        "y_train_resampled.npy"
    ),
    y_train_resampled
)

# --------------------------------------------
# 2. Save scaled test data
# --------------------------------------------

np.save(
    os.path.join(
        processed_dir,
        "X_test_scaled.npy"
    ),
    X_test_scaled
)

np.save(
    os.path.join(
        processed_dir,
        "y_test_encoded.npy"
    ),
    y_test_encoded
)

# --------------------------------------------
# 3. Verify existing preprocessing files
# --------------------------------------------

encoding_mapping_path = os.path.join(
    processed_dir,
    "encoding_mapping.json"
)

scaler_path = os.path.join(
    processed_dir,
    "scaler.pkl"
)

print("=" * 70)
print("PHASE 3 FINAL PROCESSED DATA")
print("=" * 70)

print("\nSaved / existing files:")

files_to_check = [
    "X_train_resampled.npy",
    "y_train_resampled.npy",
    "X_test_scaled.npy",
    "y_test_encoded.npy",
    "encoding_mapping.json",
    "scaler.pkl"
]

for filename in files_to_check:

    file_path = os.path.join(
        processed_dir,
        filename
    )

    print(
        f"{filename}:",
        os.path.exists(file_path)
    )

# --------------------------------------------
# 4. Display shapes
# --------------------------------------------

print("\nData shapes:")

print(
    "X_train_resampled:",
    X_train_resampled.shape
)

print(
    "y_train_resampled:",
    y_train_resampled.shape
)

print(
    "X_test_scaled:",
    X_test_scaled.shape
)

print(
    "y_test_encoded:",
    y_test_encoded.shape
)

# --------------------------------------------
# 5. Create ZIP file
# --------------------------------------------

zip_path = os.path.join(
    processed_dir,
    "processed_dataset.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for filename in files_to_check:

        file_path = os.path.join(
            processed_dir,
            filename
        )

        if os.path.exists(file_path):

            zipf.write(
                file_path,
                arcname=filename
            )

print("\n" + "=" * 70)
print("ZIP CREATED")
print("=" * 70)

print(
    "ZIP file:",
    zip_path
)

print(
    "ZIP exists:",
    os.path.exists(zip_path)
)


PHASE 3 FINAL PROCESSED DATA

Saved / existing files:
X_train_resampled.npy: True
y_train_resampled.npy: True
X_test_scaled.npy: True
y_test_encoded.npy: True
encoding_mapping.json: True
scaler.pkl: True

Data shapes:
X_train_resampled: (135441, 41)
y_train_resampled: (135441,)
X_test_scaled: (29696, 41)
y_test_encoded: (29696,)

ZIP CREATED
ZIP file: ../data/processed\processed_dataset.zip
ZIP exists: True
